In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv('.env')
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0,
    api_key=os.getenv('api_key')
)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000203A082DF10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000203A0840760>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
#### Persistance (memory)
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage, AIMessage
from langchain_tavily import TavilySearch # a searching research tool
from typing import TypedDict, Annotated, List # for State typing
import operator # for State tying
from langgraph.graph import StateGraph, END # nodes of langgraph

class AgentState(TypedDict):
    task: str
    plan: str
    draft: str
    critique: str
    content: List[str]
    revision_number: int
    max_revisions: int

In [3]:
from pydantic import BaseModel
from tavily import TavilyClient
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
import os

PLAN_PROMPT = """You are an expert writer tasked with writing a high level outline of an essay. \
Write such an outline for the user provided topic. Give an outline of the essay along with any relevant notes \
or instructions for the sections."""

RESEARCH_PLAN_PROMPT = """
You are a researcher.

Generate at most 3 search queries.

Return valid JSON only.

JSON schema:
{
  "queries": [
    "query 1",
    "query 2"
  ]
}
"""

WRITER_PROMPT = """You are an essay assistant tasked with writing excellent 5-paragraph essays.\
Generate the best essay possible for the user's request and the initial outline. \
If the user provides critique, respond with a revised version of your previous attempts. \
Utilize all the information below as needed: 

------

{content}"""

REFLECTION_PROMPT = """You are a teacher grading an essay submission. \
Generate critique and recommendations for the user's submission. \
Provide detailed recommendations, including requests for length, depth, style, etc."""

RESEARCH_CRITIQUE_PROMPT = """
You are a researcher helping revise an essay.

Generate at most 3 search queries.

Return valid JSON only.

JSON schema:
{
  "queries": [
    "query 1",
    "query 2"
  ]
}
"""

# memory
conn = sqlite3.connect("AI_Assistant_Essay_Writer.db", check_same_thread=False)
memory = SqliteSaver(conn)

prompts = {'plan_prompt':PLAN_PROMPT, 'research_plan_prompt':RESEARCH_PLAN_PROMPT, 'writer_prompt':WRITER_PROMPT, 'reflection_prompt':REFLECTION_PROMPT, 'research_critque_prompt':RESEARCH_CRITIQUE_PROMPT}

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"]) # tools you can bind or call if you are sure

class Queries(BaseModel): # pydantic Tagging input to LLM
    queries: List[str]

In [4]:
import re

class AIAssistant:
    def __init__(self, llm, memory, tavily_client_tool, prompts):
        self.model = llm
        self.prompts = prompts
        self.tavily_client_tool = tavily_client_tool

        graph = StateGraph(AgentState)
        graph.add_node('planner', self.plan_node)
        graph.add_node('research_plan', self.research_plan_node)
        graph.add_node('generate', self.generation_node)
        graph.add_node('reflect', self.reflection_node)
        graph.add_node('research_critique', self.research_critique_node)
        # Edges
        graph.set_entry_point("planner")
        graph.add_edge("planner", "research_plan")
        graph.add_edge("research_plan", "generate")
        graph.add_edge("reflect", "research_critique")
        graph.add_edge("research_critique", "generate")
        # Conditional
        graph.add_conditional_edges('generate', self.should_continue, {False: END, True: 'reflect'})
        self.graph = graph.compile(checkpointer=memory)
  
    def plan_node(self, state: AgentState):
        messages = [
            SystemMessage(content=self.prompts['plan_prompt']), 
            HumanMessage(content=state['task'])
        ]
        response = self.model.invoke(messages)
        return {"plan": self.remove_think_tag(response.content)}
    
    def research_plan_node(self, state: AgentState):
        queries = self.model.with_structured_output(Queries, method="json_mode").invoke([
            SystemMessage(content=f"{self.prompts['research_plan_prompt']}\n\nHere is my plan:\n\n{state['plan']}"),
            HumanMessage(content=state['task'])
        ])
        content = state.get("content", [])
        for q in queries.queries:
            response = self.tavily_client_tool.search(query=q, max_results=2)
            for r in response['results']:
                content.append(self.remove_think_tag(r['content']))
        return {"content": content}
    
    def generation_node(self, state: AgentState):
        content = "\n\n".join(state.get("content", []))
        user_message = HumanMessage(content=f"{state['task']}")
        messages = [
            SystemMessage(content=self.prompts['writer_prompt'].format(content=content)),
            user_message
            ]
        response = self.model.invoke(messages)
        return {
            "draft": self.remove_think_tag(response.content), 
            "revision_number": state.get("revision_number", 1) + 1
        }

    def reflection_node(self, state: AgentState):
        messages = [
            SystemMessage(content=self.prompts['reflection_prompt']), 
            HumanMessage(content=state['draft'])
        ]
        response = self.model.invoke(messages)
        return {"critique": self.remove_think_tag(response.content)}
    
    def research_critique_node(self, state: AgentState):
        queries = self.model.with_structured_output(Queries, method="json_mode").invoke([
            SystemMessage(content=self.prompts['research_critque_prompt']),
            HumanMessage(content=state['critique'])
        ])
        content = state.get("content", [])
        for q in queries.queries:
            response = self.tavily_client_tool.search(query=q, max_results=2)
            for r in response['results']:
                content.append(self.remove_think_tag(r['content']))
        return {"content": content}
    
    def should_continue(self, state: AgentState):
        return state["revision_number"] <= state["max_revisions"]
    
    def remove_think_tag(self, content):
        return re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()

essay_writer = AIAssistant(llm=llm, memory=memory, tavily_client_tool=tavily, prompts=prompts)
essay_writer

In [5]:
thread = {"configurable": {"thread_id": "1"}}
input_state = {
    'task': "what is the difference between langchain and langsmith",
    "max_revisions": 2,
    "revision_number": 1,
}

for s in essay_writer.graph.stream(input_state, thread):
    print(s)

{'planner': {'plan': "Here is a high-level outline for an essay on the difference between LangChain and LangSmith:\n\n**I. Introduction**\n* Briefly introduce the topic of AI-powered language models and their increasing importance in various industries\n* Mention LangChain and LangSmith as two notable platforms in this space\n* Thesis statement: While both LangChain and LangSmith are designed to facilitate the development and deployment of language models, they differ significantly in their approach, features, and use cases.\n\n**II. Overview of LangChain**\n* Define LangChain and its primary function as a framework for building and deploying language models\n* Discuss LangChain's key features, such as its modular architecture, support for multiple AI models, and integration with various data sources\n* Provide examples of how LangChain is used in real-world applications, such as chatbots, virtual assistants, and content generation\n\n**III. Overview of LangSmith**\n* Define LangSmith 